# 01 · Data Cleaning: Bengaluru PM2.5

**Goal:** raw CPCB `city_day.csv` se Bengaluru ka modelling-ready dataset banana.

**Output:** `data/processed/bengaluru_clean.csv` and `reports/tables/missing_before_cleaning.csv`

## 1. Imports and paths

In [21]:
import numpy as np
import pandas as pd
from pathlib import Path

# Notebook ka __file__ nahi hota, isliye current folder se project root dhoondhte hain.
# Notebook "notebooks/" ke andar se chale ya root se, dono mein ROOT = SML_AAT hi milega.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "city_day.csv"
OUT = ROOT / "data" / "processed" / "bengaluru_clean.csv"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)   # folder na ho to bana do
print("Project root:", ROOT)

Project root: C:\Users\BOBIN\Downloads\SML_AAT


## 2. Settings

- **POLLUTANTS:** jin columns ko clean karna hai.
- **MAX_GAP:** isse lambe gaps ko interpolate nahi karenge, taaki hafton ka fake data na bane.
- **UNHEALTHY_PM25:** CPCB ka 24-hour PM2.5 standard (60 µg/m³). Logistic regression ka target isi se banega.
- **CO_MAX:** isse upar ki CO reading ko sensor error maana jayega.

In [12]:
POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"]
MAX_GAP = 7          # days
UNHEALTHY_PM25 = 60  # ug/m3
CO_MAX = 10          # mg/m3, readings above this are instrument faults

## 3. Load data and filter Bengaluru

In [13]:
df = pd.read_csv(RAW, parse_dates=["Date"])
df = df[df["City"] == "Bengaluru"].sort_values("Date").reset_index(drop=True)
print(df.shape, "|", df["Date"].min().date(), "->", df["Date"].max().date())
df.head()

(2009, 16) | 2015-01-01 -> 2020-07-01


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Bengaluru,2015-01-01,NaN,NaN,3.26,17.33,10.88,20.36,0.33,3.54,10.73,0.56,4.64,NaN,NaN,NaN
1,Bengaluru,2015-01-02,NaN,NaN,6.05,19.73,14.14,23.74,1.35,3.97,22.77,0.65,5.31,NaN,NaN,NaN
2,Bengaluru,2015-01-03,NaN,NaN,11.91,19.88,20.72,4.32,17.40,13.61,12.03,0.53,19.25,NaN,NaN,NaN
3,Bengaluru,2015-01-04,NaN,NaN,7.45,21.61,16.88,0.87,5.05,6.52,17.70,0.55,7.47,NaN,NaN,NaN
4,Bengaluru,2015-01-05,NaN,NaN,9.52,22.17,21.76,31.38,1.83,4.71,12.72,0.40,4.36,NaN,NaN,NaN


## 4. Missing values BEFORE cleaning

Interpolation ke baad gaps bhar jaate hain, isliye pehle snapshot lete hain. Ye table report ke *Dataset Selection* section mein jaati hai.

In [14]:
miss = (df.drop(columns=["City", "Date"]).isna().mean() * 100).round(1).sort_values(ascending=False)
miss.rename("missing_%").to_csv(TABLES / "missing_before_cleaning.csv")
miss.to_frame("missing_%")

,missing_%
Xylene,100.0
PM10,17.9
Benzene,13.2
NH3,10.1
PM2.5,7.3
O3,7.2
AQI_Bucket,4.9
AQI,4.9
Toluene,4.6
CO,0.5


## 5. Har pollutant ka first valid date

PM10 Nov 2015 se pehle record hi nahi hua. Use constant se bharne ke bajaye sample wahin se shuru karenge.

In [15]:
df[POLLUTANTS].apply(lambda s: df.loc[s.notna(), "Date"].min().date()).to_frame("first_valid_date")

,first_valid_date
PM2.5,2015-03-20
PM10,2015-11-07
NO,2015-01-01
NO2,2015-01-01
NOx,2015-01-01
NH3,2015-01-01
CO,2015-01-01
SO2,2015-01-01
O3,2015-01-01
Benzene,2015-01-01


## 6. Drop Xylene and set the start date

In [16]:
df = df.drop(columns=["Xylene"])   # 100% missing

start = max(df.loc[df["PM2.5"].notna(), "Date"].min(),
            df.loc[df["PM10"].notna(), "Date"].min())
df = df[df["Date"] >= start].reset_index(drop=True)
print("Sample starts:", start.date(), "| rows:", len(df))

Sample starts: 2015-11-07 | rows: 1699


## 7. Remove physically impossible readings (sensor errors)

Found during regression diagnostics (notebook 04): the most influential days were sensor errors.

- **PM2.5 > PM10** is impossible, because PM2.5 is a subset of PM10. Such PM2.5 readings are set to missing.
- **CO > 10 mg/m³** is ~10× Bengaluru's normal level (median ≈ 0.9) and appears as isolated bursts, so it is treated as an instrument fault.

This is done **before** interpolation, so a bad value can never be spread into neighbouring days.

In [17]:
bad_pm = df["PM2.5"] > df["PM10"]
bad_co = df["CO"] > CO_MAX
print(f"PM2.5 > PM10: {bad_pm.sum()} days | CO > {CO_MAX}: {bad_co.sum()} days")
display(df.loc[bad_pm, ["Date", "PM2.5", "PM10"]].sort_values("PM2.5", ascending=False).head(8))
df.loc[bad_pm, "PM2.5"] = np.nan
df.loc[bad_co, "CO"] = np.nan
qc = pd.DataFrame({"rule": ["PM2.5 > PM10", f"CO > {CO_MAX} mg/m3"], "readings_removed": [int(bad_pm.sum()), int(bad_co.sum())]})
qc.to_csv(TABLES / "qc_removed_readings.csv", index=False)
qc

PM2.5 > PM10: 64 days | CO > 10: 7 days


,Date,PM2.5,PM10
609,2017-07-08,313.59,120.31
134,2016-03-20,239.54,189.80
254,2016-07-18,218.17,30.53
753,2017-11-29,197.41,64.65
768,2017-12-14,161.35,73.50
779,2017-12-25,159.52,45.31
664,2017-09-01,158.67,86.06
197,2016-05-22,152.94,43.16


,rule,readings_removed
0,PM2.5 > PM10,64
1,CO > 10 mg/m3,7


## 8. Short-gap interpolation

Time-based linear interpolation sirf ≤ 7 din ke gaps par. Lambe gaps `NaN` hi rehte hain, aur models complete rows use karenge.

In [18]:
for c in POLLUTANTS:
    s = df[c]
    filled = s.set_axis(df["Date"]).interpolate(method="time", limit_area="inside").values
    gap = s.isna()
    run_len = gap.groupby((gap != gap.shift()).cumsum()).transform("size")   # har gap kitna lamba hai
    df[c] = np.where(gap & (run_len > MAX_GAP), np.nan, filled)

df[POLLUTANTS].isna().sum().to_frame("still_missing")

,still_missing
PM2.5,50
PM10,22
NO,0
NO2,0
NOx,0
NH3,151
CO,0
SO2,0
O3,107
Benzene,114


## 9. Derived features

- **Season:** Winter (Dec-Feb), Summer (Mar-May), Monsoon (Jun-Sep), Post-monsoon (Oct-Nov)
- **Lockdown:** COVID lockdown period (25-Mar-2020 onward)
- **Unhealthy:** 1 agar PM2.5 > 60

In [19]:
def season(m):
    if m in (12, 1, 2): return "Winter"
    if m in (3, 4, 5): return "Summer"
    if m in (6, 7, 8, 9): return "Monsoon"
    return "Post-monsoon"

df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year
df["Season"] = df["Month"].map(season)
df["Lockdown"] = ((df["Date"] >= "2020-03-25") & (df["Date"] <= "2020-07-01")).astype(int)
df["Unhealthy"] = (df["PM2.5"] > UNHEALTHY_PM25).astype(int)
df[["Date", "PM2.5", "Season", "Lockdown", "Unhealthy"]].sample(5, random_state=1)

,Date,PM2.5,Season,Lockdown,Unhealthy
1548,2020-02-02,35.27,Winter,0,0
1119,2018-11-30,43.16,Post-monsoon,0,0
1165,2019-01-15,47.13,Winter,0,0
984,2018-07-18,18.83,Monsoon,0,0
1550,2020-02-04,29.27,Winter,0,0


## 10. Save and summary

In [20]:
df.to_csv(OUT, index=False)
core = ["PM2.5", "PM10", "NO2", "NOx", "NH3", "CO", "SO2", "O3"]
print("Saved:", OUT)
print("Shape:", df.shape)
print("Date range:", df["Date"].min().date(), "->", df["Date"].max().date())
print("Complete rows (core 8):", df[core].dropna().shape[0])
print(f"Unhealthy days: {df['Unhealthy'].sum()} ({df['Unhealthy'].mean():.1%})")

Saved: C:\Users\BOBIN\Downloads\SML_AAT\data\processed\bengaluru_clean.csv
Shape: (1699, 20)
Date range: 2015-11-07 -> 2020-07-01
Complete rows (core 8): 1478
Unhealthy days: 149 (8.8%)
